In [1]:
import math
from Utils.hdfs import init
from Utils.dataFrameSize import dfSizeEstimator
from pyspark.sql import SparkSession, Window, types as T, functions as F
from pyspark.storagelevel import StorageLevel

In [2]:
builder = (
    SparkSession.builder
    .appName("PlayGround")
    .master("yarn")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
)
spark = builder.getOrCreate()
init(spark)

In [ ]:
%%sql
use warehouse;

In [ ]:
%%sql
show tables;

In [ ]:
%%sql
insert into temp_flight
select * from flight limit 5000;

In [ ]:
%%sql
select count(*) from temp_flight;

In [ ]:
spark.sql("desc extended fct_flight").show(40, truncate=False)

In [ ]:
%%hdfs
ls hdfs://namenode:9000/user/hive/warehouse/warehouse.db/fct_flight/travel_date=2024-05-13

In [ ]:
%%hdfs
ls /user/jovyan

In [ ]:
### here is the main lakehouse dir "lakehouse"
### temp_flight is a subset of flight which much less cardinality of rows (approximately 1000 or so)

In [ ]:
spark.sql("""
describe extended flight;""").show(50, truncate=False)

In [ ]:
%%sql
drop table dim_passenger;

In [ ]:
%%sql
select * from dim_passenger;

In [ ]:
%%sql
select * from flight;

In [ ]:
%%sql
CREATE TABLE flight
USING DELTA
LOCATION 'hdfs:///user/jovyan/lakehouse/silver/stage_flight';

In [ ]:
%%sql
drop table temp_flight;

In [ ]:
%%sql
create table temp_flight 
using delta
location 'hdfs:///user/jovyan/lakehouse/silver/temp_flight'
as
select * from flight limit 10;

In [ ]:
%%sql
select * from temp_flight;

In [ ]:
%%sql
select * from temp_flight where aircraft_id = 'AI878';

In [ ]:
%%sql
update temp_flight set flight_cost = 15202 where aircraft_id = 'AI392';

In [ ]:
%%sql
SHOW TABLES IN default;

In [ ]:
%%sql
show tables;

In [ ]:
%%sql
delete from temp_flight where aircraft_id = 'AI490';

In [ ]:
%%sql
desc history scd_tmp_dim_aircraft;

In [ ]:
%%sql
select * from scd_tmp_dim_aircraft --version as of 0;

In [ ]:
%%sql
select count(distinct passenger_key) - count(passenger_key) from dim_passenger;

In [ ]:
%%sql
select * from scd_dim_aircraft limit 10;

In [ ]:
%%sql
select 
    aircraft_id, 
    flight_cost,
    origin_airport,
    destination_airport, 
    airplane_model, 
    distance, 
    fuel_consumed_litre, 
    avg_flight_speed_kmps, 
    engine_performance
from flight limit 3;

In [ ]:
%%sql
with possible_pr as (
    SELECT aircraft_id, max(struct(cnt, airport)).airport AS airport
    FROM (
        SELECT aircraft_id, origin_airport AS airport, COUNT(*) AS cnt
        FROM flight
        GROUP BY aircraft_id, origin_airport
    
        UNION ALL
    
        SELECT aircraft_id, destination_airport AS airport, COUNT(*) AS cnt
        FROM flight
        GROUP BY aircraft_id, destination_airport
    ) t
    GROUP BY aircraft_id)                              
select 
    f.aircraft_id, 
    pp.airport,
    f.airplane_model,
    avg(f.flight_cost) avg_flight_cost,
    sum(f.distance) total_distance, 
    avg(f.fuel_consumed_litre) avg_fuel_consumption, 
    avg(f.avg_flight_speed_kmps) avg_flight_speed_kmps, 
    max(f.engine_performance) as max_engine_performance, 
    min(f.engine_performance) as min_engine_performance
from flight f
join possible_pr pp on pp.aircraft_id = f.aircraft_id
group by 
    f.aircraft_id, 
    pp.airport,
    f.airplane_model
limit 3;

In [ ]:
%%sql
show tables;

In [ ]:
%%sql
select * from scd_dim_aircraft limit 20;

In [ ]:
%%sql var=df
select * from flight where aircraft_id = 'AI017' limit 1;

In [ ]:
df.show()

In [ ]:
row = df.toPandas()

In [ ]:
from decimal import Decimal
row.at[0, 'flight_cost'] = Decimal(10003)


In [ ]:
row.at[0, 'distance'] = 19120

In [ ]:
row.to_numpy()

In [ ]:
%%sql


with  __dbt__cte__stage_flight as (


select *
from warehouse.temp_flight
),  __dbt__cte__stg_flight as (


with raw as (
    select * from __dbt__cte__stage_flight
),

cleaned as (
    select
        -- identifiers
        flight_id,
        aircraft_id,
        itinerary_no,
        ticket_no,

        -- timestamps
        departure_time,
        arrival_time,
        travel_date,

        -- dimensions
        origin_airport,
        destination_airport,
        airplane_model,
        tail_no,
        passenger_flight_class,

        -- passenger
        passenger_name,
        passenger_country,
        passenger_dob,
        frequent_flier,
        frequent_flier_no,

        -- measures
        flight_cost,
        distance,
        turbulance,
        temp_at_dept,
        fuel_consumed_litre,
        taxi_duration_mins,
        avg_flight_speed_kmps,
        engine_performance,

        -- derived
        departure_seconds,
        uuid,

        -- flight duration in minutes
        (unix_timestamp(arrival_time) - unix_timestamp(
            concat(travel_date, ' ', departure_time)
        )) / 60.0 as flight_duration_mins,

        -- revenue per km
        case when distance > 0 then flight_cost / distance end as revenue_per_km,

        -- fuel efficiency
        case when distance > 0 then fuel_consumed_litre / distance end as fuel_per_km

    from raw
    where flight_id is not null
      and travel_date is not null
)

select * from cleaned
), stg as (
    select * from __dbt__cte__stg_flight
),

dim_passenger as (
    select * from warehouse.dim_passenger
),

dim_aircraft as (
    select aircraft_id, aircraft_key from warehouse.dim_aircraft
),

dim_airport as (
    select airport_key, airport_name from warehouse.dim_airport
),

fact as (
    select
        md5(cast(concat(coalesce(cast(s.flight_id as string), '_dbt_utils_surrogate_key_null_'), '-', coalesce(cast(s.itinerary_no as string), '_dbt_utils_surrogate_key_null_'), '-', coalesce(cast(s.ticket_no as string), '_dbt_utils_surrogate_key_null_')) as string)) as flight_key,

        -- foreign keys
        dp.passenger_key,
        da.aircraft_key,
        dap_orig.airport_key as origin_airport_key,
        dap_dest.airport_key as destination_airport_key,

        -- dates
        s.travel_date,
        year(s.travel_date) as travel_year,
        month(s.travel_date) as travel_month,
        dayofweek(s.travel_date) as travel_dow,

        -- timestamps
        s.departure_time,
        s.arrival_time,
        s.departure_seconds,

        -- flight details
        s.flight_id,
        s.itinerary_no,
        s.ticket_no,
        s.passenger_flight_class,
        s.frequent_flier,
        s.frequent_flier_no,

        -- measures
        s.flight_cost as revenue,
        s.distance,
        s.turbulance,
        s.temp_at_dept,
        s.fuel_consumed_litre,
        s.taxi_duration_mins,
        s.avg_flight_speed_kmps,
        s.engine_performance,
        s.flight_duration_mins,
        s.revenue_per_km,
        s.fuel_per_km,

        -- metadata
        s.uuid as source_uuid,
        current_timestamp() as dbt_updated_at

    from stg s
    left join dim_passenger dp
        on dp.passenger_name = s.passenger_name
        and dp.passenger_country = s.passenger_country
        and dp.passenger_dob = s.passenger_dob
    left join dim_aircraft da
        on da.aircraft_id = s.aircraft_id
    left join dim_airport dap_orig
        on dap_orig.airport_name = s.origin_airport
    left join dim_airport dap_dest
        on dap_dest.airport_name = s.destination_airport
)

select * from fact

where travel_date >= (select max(travel_date) from warehouse.fct_flight);


In [ ]:
spark.read.format('csv').option('inferSchema', 'true').option('header', 'true').load()

In [4]:
df = spark.read.table('warehouse.flight_raw')

In [6]:
df.limit(5000).coalesce(1).write.format('csv').mode('overwrite').save('sample_flight.csv')

In [8]:
%%hdfs
rm -r -f /user/jovyan/sample_flight.csv

✅ Deleted: /user/jovyan/sample_flight.csv


In [ ]:
spark.stop()